In [1]:
import pandas as pd

In [2]:
# die ganzen Imports
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import KMeans

In [3]:
# train.csv einlesen
train_file_path = "train.csv"
f1_data = pd.read_csv(train_file_path)

In [4]:
# y zuteilen --> PitNextLap
y = f1_data.PitNextLap

In [5]:
#Rausfinden welche verschiedene reifen es gibt
print("Reifen im Trainingsset:", f1_data["Compound"].unique())

Reifen im Trainingsset: <StringArray>
['HARD', 'MEDIUM', 'INTERMEDIATE', 'SOFT', 'WET']
Length: 5, dtype: str


In [6]:
#Text-Spalten in eindeutige Zahlen-Codes umwandeln
f1_data["Race_Code"] = f1_data["Race"].astype("category").cat.codes
f1_data["Driver_Code"] = f1_data["Driver"].astype("category").cat.codes

#Mapping definieren und die Spalte berechnen
reifen_mapping = {
    "HARD": 0,
    "MEDIUM": 1,
    "SOFT": 2,
    "INTERMEDIATE": 3,
    "WET": 4
}
f1_data["Compound_Code"] = f1_data["Compound"].map(reifen_mapping).fillna(-1)

#Neues Feature berechnen mit höchsten Mi-Scores
f1_data["Stint_Progress"] = f1_data["RaceProgress"] * f1_data["Stint"]


In [7]:
# feature erstellen und X zuteilen
features = [
    "Stint",
    "TyreLife",
    "Position",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "LapTime_Delta",
    "Stint_Progress",  #neuer Fortschritts-Faktor ermittelt durch MI-Score
    "Compound_Code", #neues Feature durch mapping
    "Driver_Code",   #Der Fahrer
    "Race_Code", #welches Rennen
]

X = f1_data[features]

In [8]:
# in Validierung und Trainigsdaten aufteilen
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

In [9]:
#MI-Scores berechnen
mi_scores = mutual_info_classif(X, y, random_state=1)

#In ein schönes DataFrame verpacken
mi_scores_df = pd.Series(mi_scores, name="MI Scores", index=X.columns)
mi_scores_df = mi_scores_df.sort_values(ascending=False)

#Ergebnisse anzeigen
print(mi_scores_df)

Stint_Progress            0.122384
Stint                     0.099027
RaceProgress              0.093369
Cumulative_Degradation    0.066537
Compound_Code             0.064739
LapTime_Delta             0.064510
TyreLife                  0.044323
Position_Change           0.039107
Race_Code                 0.019982
Driver_Code               0.009425
Position                  0.004746
Name: MI Scores, dtype: float64


In [10]:
# RandomForest Klassifizierer importieren
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [11]:
# Definiere RandomForest Klassifizierung Model --> Klassifizerung und nicht Regression da kein € Wert getestet wird sondern ein Prozentwert zwischen 0 und 1
# habe im zweiten schritt N-estimators=300 gesetzt, Standardgemäß sind 100 Bäume in meinem RF, nun habe ich die Zahl auf 300 gesetzt
# mit min_samples_leaf=3 will ich verhindern dass runden auswendig gelernt werden
rf_model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=1)
rf_model.fit(train_X, train_y)

# predict_proba liefert Wahrscheinlichkeiten für beide Antwortmöglichkeiten 0 oder 1; [:, 1] schneidet uns die Wahrscheinlichkeit für einen echten Boxenstopp (1) heraus
rf_val_probabilities = rf_model.predict_proba(val_X)[:, 1]

rf_val_auc = roc_auc_score(val_y, rf_val_probabilities)

print("Validation ROC-AUC-Score für dein RF-Model: {:.4f}".format(rf_val_auc))

### Test wegen Overfitting ###
# Vorhersagen für die Trainingsdaten generieren
train_probs = rf_model.predict_proba(train_X)[:, 1]

# Score für die Trainingsdaten berechnen
train_auc = roc_auc_score(train_y, train_probs)

# Trainingsscore
print(f"Trainings-Score:     {train_auc:.4f}")

Validation ROC-AUC-Score für dein RF-Model: 0.9358
Trainings-Score:     0.9951


In [12]:
# test.csv einlesen
test_data_path = "test.csv"
test_data = pd.read_csv(test_data_path)

In [13]:
#Die Interaktion zwischen Rennfortschritt und Stint für test berechnen
test_data["Stint_Progress"] = test_data["RaceProgress"] * test_data["Stint"]

#Text-Spalten in eindeutige Zahlen-Codes umwandeln
test_data["Race_Code"] = test_data["Race"].astype("category").cat.codes
test_data["Driver_Code"] = test_data["Driver"].astype("category").cat.codes

#Mapping definieren und die Spalte berechnen
reifen_mapping = {
    "HARD": 0,
    "MEDIUM": 1,
    "SOFT": 2,
    "INTERMEDIATE": 3,
    "WET": 4
}
test_data["Compound_Code"] = test_data["Compound"].map(reifen_mapping).fillna(-1)


# test_X erstellen und es werden nur die Spalten genommen von dem feature
test_X = test_data[features]

In [14]:
# Vorhersage machen die wir hochladen
rf_model_on_full_data = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=1)

# auf allen Daten trainieren
rf_model_on_full_data.fit(X, y)

# Wahrscheinlichkeit für die Testdaten(X_test_ berechnen
test_preds = rf_model_on_full_data.predict_proba(test_X)[:, 1]

In [15]:
# Tabelle so aufbauen dass ich sie submitten kann
output = pd.DataFrame({"id": test_data.id, "PitNextLap": test_preds})

# Speichert das Ganze als submission.csv
output.to_csv("submission.csv", index=False)